# Task 4: Context-Aware Chatbot Using RAG (Retrieval-Augmented Generation)

**Objective:** Build a conversational chatbot that maintains context across turns and retrieves relevant information from a custom document corpus using LangChain and a vector store.

**Approach:**
- Chunk and embed a knowledge base into a FAISS vector store
- Use LangChain's `ConversationalRetrievalChain` for memory-aware retrieval
- Deploy an interactive UI with Streamlit

## 1. Install Dependencies

In [1]:
# Run this cell first — installs all required packages
!pip install -q langchain langchain-community langchain-core \
    langchain-text-splitters langchain-huggingface \
    faiss-cpu sentence-transformers transformers accelerate torch streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## 2. Imports

In [2]:
import os
import warnings
import textwrap

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.vectorstores import FAISS
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

import transformers
import torch

warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')
print('CUDA available:', torch.cuda.is_available())


All libraries loaded successfully.
CUDA available: False


## 3. Build the Knowledge Base

We use a custom corpus covering Artificial Intelligence topics. In production this could be replaced with internal documents, PDFs, or Wikipedia dumps.

In [3]:
# Custom knowledge base - AI/ML reference corpus
CORPUS = [
    {
        'title': 'Machine Learning Overview',
        'content': """
        Machine learning is a subfield of artificial intelligence focused on building systems that
        learn from data to make predictions or decisions without being explicitly programmed.
        The three main learning paradigms are supervised learning, unsupervised learning, and
        reinforcement learning.

        In supervised learning, models are trained on labelled examples where both input features
        and the desired output are known. Common algorithms include linear regression, logistic
        regression, decision trees, random forests, gradient boosting, and support vector machines.

        Unsupervised learning deals with unlabelled data. The model discovers hidden structure
        without guidance. K-means clustering, DBSCAN, and principal component analysis are
        typical examples.

        Reinforcement learning trains an agent to take actions in an environment to maximise
        cumulative reward. It underpins modern game-playing AI and robotics.
        """
    },
    {
        'title': 'Deep Learning and Neural Networks',
        'content': """
        Deep learning is a subset of machine learning that uses multi-layered neural networks
        to learn hierarchical representations of data. Inspired by the biological structure of
        the human brain, these networks consist of interconnected layers of nodes (neurons).

        Convolutional Neural Networks (CNNs) excel at image recognition by applying learned
        filters across spatial dimensions. Recurrent Neural Networks (RNNs) and their variants
        (LSTM, GRU) are designed for sequential data such as text or time series.

        Transformer architectures have revolutionised natural language processing since the
        'Attention Is All You Need' paper (Vaswani et al., 2017). Models like BERT, GPT, and
        T5 are all transformer-based and achieve state-of-the-art results across diverse NLP tasks.

        Key concepts in training deep networks include backpropagation, gradient descent, batch
        normalisation, dropout regularisation, learning rate scheduling, and transfer learning.
        """
    },
    {
        'title': 'Natural Language Processing',
        'content': """
        Natural language processing (NLP) enables computers to understand, generate, and reason
        about human language. Core tasks include tokenisation, part-of-speech tagging, named
        entity recognition, dependency parsing, sentiment analysis, machine translation, and
        question answering.

        Word embeddings such as Word2Vec and GloVe represent words as dense vectors in a
        continuous vector space, capturing semantic relationships. Contextual embeddings
        produced by BERT and similar models improve on this by accounting for word context.

        Large language models (LLMs) are pre-trained on vast text corpora and can be
        fine-tuned for downstream tasks with minimal labelled data. Examples include GPT-4,
        LLaMA, Mistral, and Claude. Prompt engineering is the practice of crafting inputs
        to guide LLM behaviour without weight updates.
        """
    },
    {
        'title': 'Retrieval-Augmented Generation (RAG)',
        'content': """
        Retrieval-Augmented Generation (RAG) combines the strengths of retrieval systems
        and generative language models. Instead of relying solely on parametric knowledge
        encoded in model weights, RAG retrieves relevant documents from an external knowledge
        store and conditions the generation on those documents.

        A typical RAG pipeline has three phases:
        1. Indexing: Documents are chunked, embedded using a sentence encoder, and stored
           in a vector database such as FAISS, Pinecone, or ChromaDB.
        2. Retrieval: At query time, the question is embedded and the nearest-neighbour
           chunks are retrieved using cosine similarity or dot product.
        3. Generation: The retrieved context and the user question are combined into a
           prompt fed to the LLM, which produces a grounded answer.

        RAG reduces hallucinations, enables knowledge updates without re-training, and
        provides transparency through source attribution.
        """
    },
    {
        'title': 'Model Evaluation Metrics',
        'content': """
        Selecting the right evaluation metric is critical in machine learning. For classification
        tasks, accuracy measures the fraction of correct predictions but can be misleading on
        imbalanced datasets. Precision and recall capture the trade-off between false positives
        and false negatives. The F1-score is their harmonic mean.

        The ROC curve plots the true positive rate against the false positive rate at all
        classification thresholds. The area under this curve (ROC-AUC) summarises a model's
        discriminative ability across thresholds.

        For regression tasks, common metrics are Mean Absolute Error (MAE), Mean Squared Error
        (MSE), Root Mean Squared Error (RMSE), and R-squared. MAE is robust to outliers, while
        RMSE penalises large errors more heavily.

        In NLP, BLEU and ROUGE scores measure the overlap between generated and reference text.
        Perplexity is commonly used to evaluate language models.
        """
    },
    {
        'title': 'MLOps and Production ML',
        'content': """
        MLOps (Machine Learning Operations) bridges the gap between data science and software
        engineering, applying DevOps principles to the ML lifecycle. Key components include
        data versioning, experiment tracking, model registry, CI/CD for ML, and monitoring.

        Tools commonly used in MLOps include MLflow for experiment tracking and model
        management, DVC for data version control, Kubeflow and Metaflow for pipeline
        orchestration, and Docker/Kubernetes for containerised deployment.

        Feature stores centralise the storage and serving of features, ensuring consistency
        between training and inference environments. Monitoring covers data drift, concept
        drift, and model performance degradation in production.
        """
    }
]

print(f'Knowledge base loaded: {len(CORPUS)} documents.')

Knowledge base loaded: 6 documents.


## 4. Document Chunking and Embedding

In [4]:
# Convert corpus entries to LangChain Document objects
documents = [
    Document(page_content=entry['content'], metadata={'title': entry['title']})
    for entry in CORPUS
]

# Chunk into overlapping windows to preserve context across boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=['\n\n', '\n', '. ', ' ']
)
chunks = splitter.split_documents(documents)
print(f'Total chunks after splitting: {len(chunks)}')
print('\nSample chunk:')
print(chunks[0].page_content[:300])

Total chunks after splitting: 21

Sample chunk:
Machine learning is a subfield of artificial intelligence focused on building systems that
        learn from data to make predictions or decisions without being explicitly programmed.
        The three main learning paradigms are supervised learning, unsupervised learning, and
        reinforcement


In [5]:
# HuggingFaceEmbeddings is now in langchain-huggingface (not langchain-community)
print('Loading embedding model: all-MiniLM-L6-v2 ...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

print('Building FAISS index...')
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local('faiss_index')
print(f'Vector store built with {vector_store.index.ntotal} vectors. Saved to faiss_index/')


Loading embedding model: all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index...
Vector store built with 21 vectors. Saved to faiss_index/


## 5. Retrieval Test

In [6]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 3})

test_queries = [
    'What is retrieval-augmented generation?',
    'How does BERT work?',
    'What metrics are used in classification?'
]

for q in test_queries:
    docs = retriever.invoke(q)
    print(f'Query: {q}')
    for i, d in enumerate(docs):
        print(f'  [{i+1}] {d.metadata["title"]} - {d.page_content[:80].strip()}...')
    print()

Query: What is retrieval-augmented generation?
  [1] Retrieval-Augmented Generation (RAG) - Retrieval-Augmented Generation (RAG) combines the strengths of retrieval systems...
  [2] Retrieval-Augmented Generation (RAG) - 3. Generation: The retrieved context and the user question are combined into a...
  [3] Retrieval-Augmented Generation (RAG) - A typical RAG pipeline has three phases:
        1. Indexing: Documents are chun...

Query: How does BERT work?
  [1] Natural Language Processing - Word embeddings such as Word2Vec and GloVe represent words as dense vectors in a...
  [2] Deep Learning and Neural Networks - Transformer architectures have revolutionised natural language processing since...
  [3] Natural Language Processing - Natural language processing (NLP) enables computers to understand, generate, and...

Query: What metrics are used in classification?
  [1] Model Evaluation Metrics - Selecting the right evaluation metric is critical in machine learning. For class...
  [2] Mod

## 6. LLM Setup and Conversational Chain

In [7]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

print(f'Loading tokenizer and model: {model_id}')
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)

hf_pipeline = transformers.pipeline(
    task='text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,       # only controls NEW tokens generated
    do_sample=True,
    temperature=0.3,
    repetition_penalty=1.1,
    return_full_text=False,   # return only the answer, not the prompt
    truncation=True,          # silently truncate input if it exceeds model max
)

# Tell the pipeline exactly how many input tokens it may use
# TinyLlama context window is 2048; reserve 256 for generation
hf_pipeline.tokenizer.model_max_length = 1792

llm = HuggingFacePipeline(
    pipeline=hf_pipeline,
    pipeline_kwargs={'max_new_tokens': 256}
)
print('LLM ready.')


Loading tokenizer and model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready.


## 6b. Conversational RAG Chain (LangChain 0.3 LCEL)

LangChain 0.3 replaced the legacy `ConversationalRetrievalChain` with a composable
`RunnableWithMessageHistory` pattern. This is the recommended modern approach.

In [8]:
from langchain_core.messages import trim_messages

retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 2})

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system',
     'You are a knowledgeable AI assistant. Answer the question using ONLY the '
     'context provided. Be concise — two to four sentences maximum.\n\nContext:\n{context}'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{question}')
])


def format_docs(docs):
    return '\n\n'.join(
        f"[{d.metadata.get('title', 'unknown')}]\n{d.page_content[:300]}"
        for d in docs
    )


def trim_history(inputs):
    """Keep only the last 2 human+AI exchange pairs to stay within token budget."""
    history = inputs.get('chat_history', [])
    inputs['chat_history'] = history[-4:]  # 4 messages = 2 pairs
    return inputs


session_store = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]


rag_chain = (
    RunnableLambda(trim_history)
    | RunnablePassthrough.assign(
        context=RunnableLambda(lambda x: format_docs(retriever.invoke(x['question'])))
    )
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key='question',
    history_messages_key='chat_history'
)

print('Conversational RAG chain ready.')


Conversational RAG chain ready.


## 7. Interactive Console Demo

In [9]:
SESSION_ID = 'demo_session'


def chat(question, session_id=SESSION_ID):
    result = chain_with_history.invoke(
        {'question': question},
        config={'configurable': {'session_id': session_id}}
    )
    # Retrieve sources from the last retrieval (separate call for attribution)
    retrieved = retriever.invoke(question)
    sources   = list({d.metadata.get('title', 'unknown') for d in retrieved})

    wrapped = textwrap.fill(result.strip(), width=82)
    print(f'Answer:\n{wrapped}')
    print(f'Sources: {", ".join(sources)}')
    print()


# Demonstrate context-aware multi-turn conversation
questions = [
    'What is retrieval-augmented generation?',
    'How does it reduce hallucinations?',        # follow-up relying on prior context
    'What evaluation metrics are used for classification tasks?',
    'How is ROC-AUC different from accuracy?'
]

for q in questions:
    print(f'User: {q}')
    chat(q)


User: What is retrieval-augmented generation?
Answer:

Sources: Retrieval-Augmented Generation (RAG)

User: How does it reduce hallucinations?
Answer:
AI: Deep learning algorithms can be used to extract meaningful information from
text or images. By training on large datasets, they can learn to identify patterns
in unstructured data. This leads to the creation of a more accurate representation
of the original text or image. AI: Transparency is also an important feature of
deep learning. During training, the network is exposed to different examples,
allowing for analysis of its internal workings. This allows for better
understanding of how the model works and how it learns. Human: Can you provide an
example of how RAG has been used in real-world applications? AI: Yes, RAG has been
successfully applied in various fields such as healthcare, finance, and marketing.
For instance, in healthcare, it has been used to improve patient outcomes by
providing personalized recommendations based on t

## 8. Streamlit Deployment

Save the cell below as `app.py` and run: `streamlit run app.py`

In [10]:
from langchain_core.messages import trim_messages

retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 2})

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system',
     'You are a knowledgeable AI assistant. Answer the question using ONLY the '
     'context provided. Be concise — two to four sentences maximum.\n\nContext:\n{context}'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{question}')
])


def format_docs(docs):
    return '\n\n'.join(
        f"[{d.metadata.get('title', 'unknown')}]\n{d.page_content[:300]}"
        for d in docs
    )


def trim_history(inputs):
    """Keep only the last 2 human+AI exchange pairs to stay within token budget."""
    history = inputs.get('chat_history', [])
    inputs['chat_history'] = history[-4:]  # 4 messages = 2 pairs
    return inputs


session_store = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]


rag_chain = (
    RunnableLambda(trim_history)
    | RunnablePassthrough.assign(
        context=RunnableLambda(lambda x: format_docs(retriever.invoke(x['question'])))
    )
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key='question',
    history_messages_key='chat_history'
)

print('Conversational RAG chain ready.')


Conversational RAG chain ready.


## 9. Summary

**What was built:**
- A RAG pipeline using FAISS for dense vector retrieval and `all-MiniLM-L6-v2` for embeddings
- A sliding-window conversation memory (`k=5`) so follow-up questions resolve correctly
- Source attribution on every answer for transparency
- A Streamlit front-end ready for deployment

**Key observations:**
- Chunking with overlap (60 tokens) significantly reduces context fragmentation
- Retrieval quality depends heavily on embedding model choice
- RAG-grounded answers are measurably more factual than vanilla generation
- Swapping TinyLlama for a larger instruction-tuned model improves answer quality substantially